In [35]:
#import google.generativeai as genai
import pandas as pd
import json
import re
import time
from pathlib import Path
from pdf2image import convert_from_path
from IPython.display import display, Markdown
from google import genai
from google.genai.types import HttpOptions
from google.oauth2.credentials import Credentials

import mimetypes

#from tkinter.filedialog import askopenfilename   # File picker

# --- CONFIGURATION ---

base_url = "https://vertexai.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7/"
access_token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJhZXNKN2kxNGNidnVuTU40MTJrOU5yZ2ROeENhTlJudTNPbC1TU08ycFlJIn0.eyJleHAiOjE3NjQ4NDU3OTIsImlhdCI6MTc2NDg0Mzk5NCwiYXV0aF90aW1lIjoxNzY0ODQzOTkxLCJqdGkiOiJlMGJmMjJhYS1kNTg0LTRmZWEtODM2ZC1lYTJmMmFjNjU1NWMiLCJpc3MiOiJodHRwczovL2F1dGgubWNraW5zZXkuaWQvYXV0aC9yZWFsbXMvciIsImF1ZCI6ImJjZDIzNzI4LTNkMjctNDQ3Yy1hMGE5LWVhY2FmMzkzYTZmNSIsInN1YiI6IjI0NDRiYzZjLTAwMzctNGIyZS1hYzI3LWZjNTlhNTkxNTM2NiIsInR5cCI6IklEIiwiYXpwIjoiYmNkMjM3MjgtM2QyNy00NDdjLWEwYTktZWFjYWYzOTNhNmY1Iiwic2Vzc2lvbl9zdGF0ZSI6IjE5ODE4ZWIwLWUwZjEtNDUyOC1iYjJiLTNhNWQzNTcyZTA2OCIsImF0X2hhc2giOiJabE0yLUVhelNCQjcyUFBLLXpHbWZRIiwibmFtZSI6IlVnYW5kaGFyIFZhZGRpIiwiZ2l2ZW5fbmFtZSI6IlVnYW5kaGFyIiwiZmFtaWx5X25hbWUiOiJWYWRkaSIsInByZWZlcnJlZF91c2VybmFtZSI6IjE1ZDhiYmNkMmMzNTNmYWUiLCJlbWFpbCI6IlVnYW5kaGFyX1ZhZGRpQG1ja2luc2V5LmNvbSIsImFjciI6IjEiLCJzaWQiOiIxOTgxOGViMC1lMGYxLTQ1MjgtYmIyYi0zYTVkMzU3MmUwNjgiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZm1ubyI6IjM0NzI3NiIsImdyb3VwcyI6WyI3YTRmM2Q2My1iNWRiLTRkNWItODA3Ni04ZjExNGQxZjE0ZjciLCJBbGwgRmlybSBVc2VycyJdfQ.HrGdbF02pzH-775ERSgCDAppDQAd2UfeEygtstaLq8DzPzc3v80JFY42hQoGa_0TamoVCP8bYpQb_dJZplMQtg5dzYbiRpqdykrJKzc6J8j8lifj8LeRr2DrVoP4ka4rNnV-2Y6Dym5uQ1MEx3MxzdhSFG63hcP9QF7GG00tCnCGzKsqDBhNXP4SJEERgreew3PRnHZzrr5xxPilc6xAFBnSYxlN1WMVtzOXhEEENpHPe2vCeEF-x1uenKCq0GVSTJxDyqrV1_jqQC1GtUv1Zso_yIJVhplKGDLDJ6jEnkk1e8iQgezZ3ii47VE28i3LokK2zvtQPVZqEwnh-_oIFA"
credentials = Credentials(access_token)

client = genai.Client(
    http_options=HttpOptions(
        api_version="v1",
        base_url=base_url,
    ),
    vertexai=True,
    project="aigateway",
    location="global",
    credentials=credentials,
)
# model = genai.GenerativeModel('models/gemini-2.5-pro')
# print(f"AI Configured: {model.model_name}")

# --- UNIVERSAL PROMPT ---
def get_robust_prompt():
    return """
    You are an advanced Engineering Data Extraction AI.
    Analyze the attached technical drawing (which could be Mechanical, Electrical, Civil, or Architectural).

    YOUR TASK:
    Visually identify and extract ALL tabular data present in the image.
    Look for grid lines, column headers, and structured lists.

    COMMON TABLE TYPES TO EXTRACT:
    - Bill of Materials (BOM) / Parts List
    - Revision History / Revision Block
    - Technical Specifications / Data Tables
    - Legends / Symbol Keys
    - Schedules (e.g., Door, Window, Pipe, Wire)
    - Pinouts or Connector Views
    - General Notes (ONLY if structured as a numbered list/table)

    OUTPUT FORMAT:
    Return ONLY a valid JSON object. No markdown.
    Structure:
    {
      "Table_Name_1": [
        ["Header1", "Header2", "Header3"],
        ["Row1_Col1", "Row1_Col2", "Row1_Col3"]
      ],
      "Table_Name_2": [ ... ]
    }

    CRITICAL ACCURACY RULES:
    1. Exact Transcription: Copy Part Numbers, Values, Dimensions, and Codes exactly as they appear.
    2. Null Handling: If a cell is visually empty, use null.
    3. Header Inference: If headers are missing, infer them from the data context (e.g., "Description", "Qty").
    4. Merged Cells: If a cell spans multiple columns visually, repeat the value or format it logically.
    """

def analyze_image(image_path):
    print(f"   -> Analyzing: {image_path}...")

    # Read the image as bytes
    with open(image_path, "rb") as f:
        image_bytes = f.read()

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        # fall back if unknown, since you're using PNG a lot
        mime_type = "image/png"

    image_part = types.Part.from_bytes(
        data=image_bytes,
        mime_type=mime_type,
    )

    # Call the model with prompt + image Part
    response = client.models.generate_content(
        # If this 404s via gateway, try "models/gemini-2.5-pro"
        model="gemini-2.5-pro",
        contents=[
            get_robust_prompt(),
            image_part,
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        ),
    )

    raw_text = response.text

    # Safety net if gateway ignores JSON mode and wraps in ```json
    if "```json" in raw_text:
        raw_text = raw_text.split("```json")[1].split("```")[0]
    elif "```" in raw_text:
        raw_text = raw_text.split("```")[1].split("```")[0]

    return json.loads(raw_text)


def process_file(file_path):
    print(f"\n--- Processing File: {file_path} ---")
    ext = Path(file_path).suffix.lower()
    temp_images = []

    if ext == ".pdf":
        print("   -> Converting PDF to images...")
        pages = convert_from_path(file_path, dpi=300)
        for i, p in enumerate(pages):
            img_path = f"temp_page_{i}.png"
            p.save(img_path, "PNG")
            temp_images.append(img_path)
    else:
        temp_images = [file_path]

    all_tables = {}

    for img in temp_images:
        page_data = analyze_image(img)
        if page_data:
            for k,v in page_data.items():
                name, n = k, 1
                while name in all_tables:     # handle duplicates
                    name = f"{k}_{n}"
                    n += 1
                all_tables[name] = v

    if all_tables:
        excel = f"{Path(file_path).stem}_Tables.xlsx"
        with pd.ExcelWriter(excel, engine="openpyxl") as writer:
            for sheet, data in all_tables.items():
                headers = data[0]
                rows = data[1:]
                df = pd.DataFrame(rows, columns=headers)

                clean_sheet = re.sub(r'[\\/*?:\[\]]', "", sheet)[:30]
                df.to_excel(writer, sheet_name=clean_sheet, index=False)
                print(f"   -> Saved sheet: {clean_sheet}")

        print(f"\nSUCCESS! Saved: {excel}")

    else:
        print("No tables found.")

    if ext == ".pdf":
        for t in temp_images:
            Path(t).unlink(missing_ok=True)

print("\nUniversal Extractor Ready.")


Universal Extractor Ready.


In [36]:
file_path = "Data Extraction/Sanitized sheet 1 (var 2).pdf"
process_file(file_path)


--- Processing File: Data Extraction/Sanitized sheet 1 (var 2).pdf ---
   -> Converting PDF to images...
No tables found.
